# 🏃 Tango Benchmark — **Notebook A** (Seed 0)

### What this notebook runs:
| Optimizer | LR | Seed | ~Time |
|-----------|-----|------|-------|
| Adam | 1e-3 | 0 | ~20 min |
| Adam | 3e-4 | 0 | ~20 min |
| Adam | 1e-4 | 0 | ~20 min |
| Lion | 1e-4 *(best LR — update below if needed)* | 0 | ~20 min |
| Tango | — | 0 | ~22 min |

**Total ≈ 1.7 hours on a T4 GPU.**

### Instructions:
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells top-to-bottom.
3. At the end, **copy the printed JSON block** into a text file called `results_A.txt`.
4. Share `results_A.txt` with the person running the **Combiner notebook**.

> **Note on Lion LR**: The Lion LR below is set to `1e-4` as a sensible default.  
> After all 4 notebooks finish, the Combiner will pick the true best Adam LR automatically.

In [ ]:
# ── Cell 1: Install deps & check GPU ──────────────────────────────────────────
!pip install -q torch torchvision
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only ⚠️')
print('CUDA:', torch.version.cuda)

In [ ]:
# ── Cell 2: Imports & shared config ───────────────────────────────────────────
import copy, json, math, os, time, urllib.request
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as transforms

# ── Reproducibility seed list for this notebook ────────────────────────────
THIS_SEED = 0

# ── Training config ────────────────────────────────────────────────────────
N_EPOCHS   = 50
BATCH_SIZE = 128
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOG_EVERY  = 1   # log every epoch

# ── Adam LR grid for THIS seed ────────────────────────────────────────────
ADAM_LRS   = [1e-3, 3e-4, 1e-4]

# ── Lion LR (default best; override if you know better after other notebooks) ─
LION_LR    = 1e-4

print(f'Config: {N_EPOCHS} epochs | batch={BATCH_SIZE} | device={DEVICE}')
print(f'Will run: Adam×{len(ADAM_LRS)} + Lion×1 + Tango×1  (all seed={THIS_SEED})')

In [ ]:
# ── Cell 3: CIFAR-10 data loaders ─────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

def make_loaders(seed):
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,
                                               shuffle=True,  num_workers=2, generator=g)
    test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256,
                                               shuffle=False, num_workers=2)
    return train_loader, test_loader

print(f'CIFAR-10 ready | train={len(train_set)} | test={len(test_set)}')

In [ ]:
# ── Cell 4: ResNet-18 model ────────────────────────────────────────────────────
# Using torchvision's ResNet-18, adapted for CIFAR-10 (32×32 input)
def make_resnet18():
    model = torchvision.models.resnet18(weights=None)
    # CIFAR adaptation: smaller first conv, no maxpool
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(512, 10)
    return model.to(DEVICE)

# Quick param count check
_tmp = make_resnet18()
print(f'ResNet-18 params: {sum(p.numel() for p in _tmp.parameters())/1e6:.2f}M')
del _tmp

In [ ]:
# ── Cell 5: Lion optimizer (pure PyTorch) ─────────────────────────────────────
class Lion(torch.optim.Optimizer):
    """Lion optimizer (Chen et al., 2023) — sign-based momentum update."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                exp_avg = state['exp_avg']
                beta1, beta2 = group['betas']
                # weight decay
                if group['weight_decay'] != 0:
                    p.data.mul_(1 - group['lr'] * group['weight_decay'])
                # update
                update = exp_avg * beta1 + grad * (1 - beta1)
                p.add_(torch.sign(update), alpha=-group['lr'])
                # update ema
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)
        return loss

print('Lion optimizer defined ✓')

In [ ]:
# ── Cell 6: Tango optimizer ────────────────────────────────────────────────────
# Tango hyperparams (from benchmark.py)
LR_MAX           = 0.005
LR_MIN           = 0.000001
T_CYCLE          = 800
EXPLORE_FRAC     = 0.8
LR_EXPLOIT_START = 0.005
LR_EXPLOIT_END   = 0.0005
EXPLORE_DECAY_START = 0.6
EXPLORE_FLOOR       = 0.03
TANG_BETA        = 0.80
TANG_EPS_BASE    = 4e-4
TANG_INTERVAL    = 100
TANG_LOSS_GATE   = 0
SHARP_UPDATE_INT = 200
FD_EPS           = 1e-4
NOISE_SCALE      = 2e-4
NOISE_START      = 500

def cyclic_lr(step, T, lr_max, lr_min):
    t = step % T
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * t / T))

def cosine_decay_lr(step, total_steps, lr_start, lr_end):
    progress = min(step / max(total_steps, 1), 1.0)
    return lr_end + 0.5 * (lr_start - lr_end) * (1 + np.cos(np.pi * progress))

def get_exploration_factor(step, total_steps, decay_start_frac, floor):
    decay_start = int(total_steps * decay_start_frac)
    if step < decay_start: return 1.0
    dp = (step - decay_start) / max(total_steps - decay_start, 1)
    return max(floor, 1.0 - dp)

def get_flat_grad(model):
    grads = []
    for p in model.parameters():
        if p.grad is not None:
            grads.append(p.grad.detach().cpu().float().view(-1).numpy())
        else:
            grads.append(np.zeros(p.numel(), dtype=np.float32))
    return np.concatenate(grads).astype(np.float32)

def get_flat_params(model):
    return np.concatenate(
        [p.detach().cpu().float().view(-1).numpy() for p in model.parameters()]
    ).astype(np.float32)

def set_flat_params(model, flat):
    offset = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(torch.tensor(flat[offset:offset+n], dtype=p.dtype,
                                   device=p.device).view(p.shape))
        offset += n

def fd_curvature(model, xb, yb, g_np, eps=FD_EPS):
    norm_g = np.linalg.norm(g_np)
    if norm_g < 1e-12: return 0.0
    g_hat   = (g_np / norm_g).astype(np.float32)
    params0 = get_flat_params(model)
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        f0 = criterion(model(xb), yb).item()
    set_flat_params(model, params0 + g_hat * eps)
    with torch.no_grad():
        f_eps = criterion(model(xb), yb).item()
    set_flat_params(model, params0)
    return float((f_eps - f0 - eps * norm_g) / (0.5 * eps**2 + 1e-30))

class TangentMomentum:
    def __init__(self, dim, beta=TANG_BETA):
        self.beta = beta
        self.v    = np.zeros(dim, dtype=np.float32)
    def step(self, model, g_np, epsilon):
        g_hat = g_np / (np.linalg.norm(g_np) + 1e-8)
        noise = np.random.randn(len(g_np)).astype(np.float32)
        noise -= np.dot(noise, g_hat) * g_hat
        noise /= np.linalg.norm(noise) + 1e-8
        self.v     = self.beta * self.v + (1 - self.beta) * noise
        direction  = self.v / (np.linalg.norm(self.v) + 1e-8)
        params0    = get_flat_params(model)
        set_flat_params(model, params0 + direction * epsilon)

print('Tango helpers defined ✓')

In [ ]:
# ── Cell 7: Evaluation helper ─────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out  = model(xb)
        loss = criterion(out, yb)
        total_loss += loss.item() * yb.size(0)
        correct    += (out.argmax(1) == yb).sum().item()
        total      += yb.size(0)
    model.train()
    return total_loss / total, correct / total

print('evaluate() helper ready ✓')

In [ ]:
# ── Cell 8: Run Adam (one LR, one seed) ───────────────────────────────────────
def run_adam(seed, lr):
    torch.manual_seed(seed); np.random.seed(seed)
    train_loader, test_loader = make_loaders(seed)
    model     = make_resnet18()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()
    history   = []
    best_acc  = 0.0
    best_loss = float('inf')
    t0 = time.time()
    print(f'  → Adam lr={lr:.0e} seed={seed} starting...')
    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        ep_loss = 0.0; ep_n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item() * yb.size(0); ep_n += yb.size(0)
        val_loss, val_acc = evaluate(model, test_loader)
        if val_loss < best_loss:
            best_loss = val_loss; best_acc = val_acc
        history.append({'epoch': epoch, 'train_loss': ep_loss/ep_n,
                        'val_loss': val_loss, 'val_acc': val_acc})
        if epoch % 10 == 0:
            print(f'    epoch {epoch:3d}/{N_EPOCHS}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')
    elapsed = time.time() - t0
    print(f'  ✓ Done in {elapsed/60:.1f} min | best_val_loss={best_loss:.4f}  best_acc={best_acc:.4f}')
    return {'optimizer': 'adam', 'lr': lr, 'seed': seed,
            'final_val_loss': val_loss, 'final_val_acc': val_acc,
            'best_val_loss': best_loss, 'best_val_acc': best_acc,
            'elapsed_sec': elapsed, 'history': history}

print('run_adam() defined ✓')

In [ ]:
# ── Cell 9: Run Lion ──────────────────────────────────────────────────────────
def run_lion(seed, lr):
    torch.manual_seed(seed); np.random.seed(seed)
    train_loader, test_loader = make_loaders(seed)
    model     = make_resnet18()
    optimizer = Lion(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    history   = []
    best_acc  = 0.0
    best_loss = float('inf')
    t0 = time.time()
    print(f'  → Lion lr={lr:.0e} seed={seed} starting...')
    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        ep_loss = 0.0; ep_n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item() * yb.size(0); ep_n += yb.size(0)
        val_loss, val_acc = evaluate(model, test_loader)
        if val_loss < best_loss:
            best_loss = val_loss; best_acc = val_acc
        history.append({'epoch': epoch, 'train_loss': ep_loss/ep_n,
                        'val_loss': val_loss, 'val_acc': val_acc})
        if epoch % 10 == 0:
            print(f'    epoch {epoch:3d}/{N_EPOCHS}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')
    elapsed = time.time() - t0
    print(f'  ✓ Done in {elapsed/60:.1f} min | best_val_loss={best_loss:.4f}  best_acc={best_acc:.4f}')
    return {'optimizer': 'lion', 'lr': lr, 'seed': seed,
            'final_val_loss': val_loss, 'final_val_acc': val_acc,
            'best_val_loss': best_loss, 'best_val_acc': best_acc,
            'elapsed_sec': elapsed, 'history': history}

print('run_lion() defined ✓')

In [ ]:
# ── Cell 10: Run Tango ────────────────────────────────────────────────────────
def run_tango(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    train_loader, test_loader = make_loaders(seed)
    model     = make_resnet18()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX,
                                   betas=(0.9, 0.999), weight_decay=0.0)
    criterion = nn.CrossEntropyLoss()

    n_params  = sum(p.numel() for p in model.parameters())
    tang_mom  = TangentMomentum(dim=n_params, beta=TANG_BETA)

    # Compute total steps
    steps_per_epoch = len(train_loader)
    total_steps     = N_EPOCHS * steps_per_epoch
    explore_end     = int(total_steps * EXPLORE_FRAC)

    best_val_loss = float('inf')
    best_val_acc  = 0.0
    best_state    = copy.deepcopy(model.state_dict())
    sharpness     = 0.0
    tang_exec     = 0
    tang_block    = 0
    history       = []
    step          = 0
    t0 = time.time()

    print(f'  → Tango seed={seed} | {n_params/1e6:.2f}M params | {total_steps} steps')

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        ep_loss = 0.0; ep_n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            # LR schedule
            if step < explore_end:
                lr = cyclic_lr(step, T_CYCLE, LR_MAX, LR_MIN)
            else:
                exploit_step  = step - explore_end
                exploit_total = total_steps - explore_end
                lr = cosine_decay_lr(exploit_step, exploit_total, LR_EXPLOIT_START, LR_EXPLOIT_END)
            for pg in optimizer.param_groups: pg['lr'] = lr

            ef = get_exploration_factor(step, total_steps, EXPLORE_DECAY_START, EXPLORE_FLOOR)

            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            g_np = get_flat_grad(model)
            optimizer.step()
            loss_val = loss.item()
            ep_loss += loss_val * yb.size(0); ep_n += yb.size(0)

            # Gradient noise
            if step >= NOISE_START:
                sigma = NOISE_SCALE * np.sqrt(max(loss_val, 1e-8)) * ef
                with torch.no_grad():
                    for p in model.parameters():
                        p.data.add_(torch.randn_like(p) * sigma)

            # Sharpness
            if step % SHARP_UPDATE_INT == 0 and step > 0:
                sharpness = fd_curvature(model, xb, yb, g_np)

            # Tangent step
            if step % TANG_INTERVAL == 0 and step > 500:
                loss_gate  = loss_val > TANG_LOSS_GATE * max(best_val_loss, 1e-10)
                dyn_limit  = 0.8 * (2.0 / max(lr, 1e-8))
                sharp_gate = sharpness < dyn_limit
                if loss_gate and sharp_gate:
                    sharp_safe = max(abs(sharpness), 1.0)
                    eps_tang   = TANG_EPS_BASE * (50.0 / sharp_safe) ** 0.5
                    eps_tang   = float(np.clip(eps_tang, 1e-5, 5e-3)) * ef
                    tang_mom.step(model, g_np, epsilon=eps_tang)
                    tang_exec += 1
                else:
                    tang_block += 1

            step += 1

        val_loss, val_acc = evaluate(model, test_loader)
        if val_loss < best_val_loss:
            best_val_loss = val_loss; best_val_acc = val_acc
            best_state    = copy.deepcopy(model.state_dict())
        history.append({'epoch': epoch, 'train_loss': ep_loss/ep_n,
                        'val_loss': val_loss, 'val_acc': val_acc,
                        'lr': lr, 'tang_exec': tang_exec, 'tang_block': tang_block})
        if epoch % 10 == 0:
            tang_rate = tang_exec / max(tang_exec + tang_block, 1)
            print(f'    epoch {epoch:3d}/{N_EPOCHS}  val_loss={val_loss:.4f}  '
                  f'val_acc={val_acc:.4f}  tang_rate={tang_rate:.2%}')

    # Restore best
    model.load_state_dict(best_state)
    final_val_loss, final_val_acc = evaluate(model, test_loader)
    elapsed = time.time() - t0
    tang_rate = tang_exec / max(tang_exec + tang_block, 1)
    print(f'  ✓ Done in {elapsed/60:.1f} min | best_val_loss={best_val_loss:.4f}  '
          f'best_acc={best_val_acc:.4f}  tang_rate={tang_rate:.2%}')
    return {'optimizer': 'tango', 'seed': seed,
            'final_val_loss': final_val_loss, 'final_val_acc': final_val_acc,
            'best_val_loss': best_val_loss, 'best_val_acc': best_val_acc,
            'tang_exec': tang_exec, 'tang_block': tang_block, 'tang_rate': tang_rate,
            'elapsed_sec': elapsed, 'history': history}

print('run_tango() defined ✓')

In [ ]:
# ── Cell 11: RUN EVERYTHING (seed 0) ─────────────────────────────────────────
grand_start = time.time()
results = {'notebook': 'A', 'seed': THIS_SEED, 'adam': [], 'lion': [], 'tango': []}

print('='*60)
print(f'  NOTEBOOK A — SEED {THIS_SEED}')
print('='*60)

# Adam: all 3 LRs
print('\n── Adam (3 LRs) ──')
for lr in ADAM_LRS:
    r = run_adam(THIS_SEED, lr)
    results['adam'].append(r)

# Lion
print(f'\n── Lion (lr={LION_LR:.0e}) ──')
r = run_lion(THIS_SEED, LION_LR)
results['lion'].append(r)

# Tango
print('\n── Tango ──')
r = run_tango(THIS_SEED)
results['tango'].append(r)

results['total_elapsed_sec'] = time.time() - grand_start
print(f'\n✅ Notebook A DONE in {results["total_elapsed_sec"]/3600:.2f} hours')

In [ ]:
# ── Cell 12: Print & save results — COPY THIS OUTPUT ─────────────────────────
# Strip history to keep output compact (combiner only needs summary stats)
def slim(r):
    s = {k: v for k, v in r.items() if k != 'history'}
    # Keep last-epoch history only for plotting
    s['history_summary'] = [
        {'epoch': h['epoch'], 'val_loss': h['val_loss'], 'val_acc': h['val_acc']}
        for h in r.get('history', [])
    ]
    return s

slim_results = {
    'notebook': results['notebook'],
    'seed':     results['seed'],
    'total_elapsed_sec': results['total_elapsed_sec'],
    'adam':  [slim(r) for r in results['adam']],
    'lion':  [slim(r) for r in results['lion']],
    'tango': [slim(r) for r in results['tango']],
}

output_str = json.dumps(slim_results, indent=2)

# Save to file (downloadable from Colab Files panel)
with open('results_A.txt', 'w') as f:
    f.write(output_str)

print('\n' + '='*60)
print('  📋 COPY EVERYTHING BELOW THIS LINE INTO results_A.txt')
print('='*60)
print(output_str)
print('='*60)
print('File also saved as results_A.txt — download from Colab Files panel (left sidebar).')